In [ ]:
# %% [Cell 1] Imports & config
import base64
import hashlib
import json
import re
import uuid
from datetime import date, datetime

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.cloud import bigquery
from pypdf import PdfReader

# Run `!pip install google-cloud-aiplatform` above if not already available.
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig

PROJECT_ID = "prj-dbi-prd-1"
DATASET_ID = "ds_dbi_digitalmedia_automation"
GEMINI_MODEL = "gemini-2.5-pro"
GEMINI_LOCATION = "us-central1"  # most common region for Gemini on Vertex AI — confirm
                                   # this is actually enabled for your project; your BQ
                                   # dataset is in us-west1, which doesn't tell us anything
                                   # about Vertex AI region availability/policy
MANIFEST_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_bronze_uploadManifest_weekly"
BRONZE_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_bronze_pdfPages_weekly"
SILVER_CI_HEADLINES_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_silver_ciHeadlines_weekly"
SILVER_RETAIL_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_silver_nationalRetailReadout_weekly"

bq = bigquery.Client(project=PROJECT_ID)
vertexai.init(project=PROJECT_ID, location=GEMINI_LOCATION)


# %% [Cell 1b] Create tables if they don't exist yet — idempotent, safe to re-run every time
bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{MANIFEST_TABLE}` (
        report_date DATE,
        run_id STRING,
        file_name STRING,
        file_hash STRING,
        status STRING,         -- 'active' | 'superseded' | 'cancelled'
        uploaded_at TIMESTAMP,
        uploaded_by STRING
    )
""").result()

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{BRONZE_TABLE}` (
        report_date DATE,
        run_id STRING,
        source_pdf STRING,
        page_number INT64,
        page_type STRING,
        text_extraction_raw STRING,
        llm_extraction_raw STRING,
        page_image_png BYTES,
        prompt_version STRING,
        extracted_at TIMESTAMP
    )
""").result()

# Migration for tables created before this column existed — safe to re-run.
bq.query(f"ALTER TABLE `{BRONZE_TABLE}` ADD COLUMN IF NOT EXISTS page_image_png BYTES").result()

print("Manifest and bronze tables ready.")


# %% [Cell 1c] One-time cleanup — NOT run automatically. Call reset_tables() yourself,
# from a cell below, only if you hit a "streaming buffer" error on delete (rows written
# before this notebook switched to load jobs, or written moments ago, can get stuck
# there for a while). This drops all data in both tables, so treat it as a reset
# button for test data, not something to run against a table you actually care about.
def reset_tables():
    bq.query(f"DROP TABLE IF EXISTS `{BRONZE_TABLE}`").result()
    bq.query(f"DROP TABLE IF EXISTS `{MANIFEST_TABLE}`").result()
    print("Dropped both tables. Re-run Cell 1b to recreate them, then re-upload your files.")

# reset_tables()  # <- uncomment and run manually when needed


# %% [Cell 2] Helpers: parse date, hash file, check manifest, write manifest row

def parse_date_from_filename(filename: str):
    """MCI's convention: '..._MMDDYY.pdf' e.g. Competitive_Offer_Report_050826.pdf"""
    m = re.search(r"_(\d{2})(\d{2})(\d{2})\.pdf$", filename, re.IGNORECASE)
    if not m:
        return None
    mm, dd, yy = m.groups()
    return datetime.strptime(f"20{yy}-{mm}-{dd}", "%Y-%m-%d").date()


def verify_date_from_content(pdf_bytes: bytes, expected_date):
    """Cross-check the filename-derived date against text actually inside the PDF
    (CI Headlines page reliably contains 'COMPETITIVE INTELLIGENCE HEADLINES <Month D, YYYY>').
    Cheap: only reads a couple of pages, not the whole deck."""
    from io import BytesIO
    reader = PdfReader(BytesIO(pdf_bytes))
    for page in reader.pages[:6]:
        text = page.extract_text() or ""
        m = re.search(r"([A-Z][a-z]+ \d{1,2}, 20\d{2})", text)
        if m:
            found = datetime.strptime(m.group(1), "%B %d, %Y").date()
            return found == expected_date, found
    return None, None  # couldn't verify — not necessarily a problem, just flag as unverified


def file_hash(pdf_bytes: bytes) -> str:
    return hashlib.sha256(pdf_bytes).hexdigest()


def get_active_manifest_row(report_date):
    query = f"""
        SELECT run_id, file_name, file_hash, uploaded_at
        FROM `{MANIFEST_TABLE}`
        WHERE report_date = @report_date AND status = 'active'
        ORDER BY uploaded_at DESC
        LIMIT 1
    """
    job = bq.query(
        query,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("report_date", "DATE", report_date)]
        ),
    )
    rows = list(job.result())
    return rows[0] if rows else None


def get_latest_run_id(report_date):
    """Look up the current active run_id for a report_date directly, instead of
    copy-pasting one out of earlier log output — which is exactly what caused the
    last two rounds of "nothing happened" confusion."""
    row = get_active_manifest_row(report_date)
    if row is None:
        print(f"No active manifest row for {report_date}. Check show_bronze_state() or upload the file first.")
        return None
    print(f"Using run_id {row['run_id']} (uploaded {row['uploaded_at']}, file {row['file_name']!r})")
    return row["run_id"]


def write_row(table_id, row: dict):
    """Load-job insert instead of streaming (insert_rows_json) — load jobs write
    directly to the table with no streaming buffer, so a DELETE run shortly after
    (e.g. while testing) won't hit BigQuery's 'can't DML rows in streaming buffer'
    error."""
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    )
    job = bq.load_table_from_json([row], table_id, job_config=job_config)
    job.result()  # raises on failure


def write_rows(table_id, rows: list):
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    )
    job = bq.load_table_from_json(rows, table_id, job_config=job_config)
    job.result()


def write_manifest_row(report_date, run_id, file_name, hash_, status):
    row = {
        "report_date": report_date.isoformat(),
        "run_id": run_id,
        "file_name": file_name,
        "file_hash": hash_,
        "status": status,
        "uploaded_at": datetime.utcnow().isoformat(),
        "uploaded_by": "notebook1",
    }
    write_row(MANIFEST_TABLE, row)


def delete_existing_run(report_date) -> bool:
    """Literal delete-then-replace, as requested — removes both the old bronze rows
    and the old manifest row for this report_date before the new run writes its own.
    Returns False if it couldn't complete, so the caller can abort rather than write
    new data on top of old data that failed to clear."""
    for table_id in (BRONZE_TABLE, MANIFEST_TABLE):
        try:
            bq.query(
                f"DELETE FROM `{table_id}` WHERE report_date = @report_date",
                job_config=bigquery.QueryJobConfig(
                    query_parameters=[bigquery.ScalarQueryParameter("report_date", "DATE", report_date)]
                ),
            ).result()
        except Exception as e:
            if "streaming buffer" in str(e):
                print(f"Can't delete from {table_id} yet — those rows are still in BigQuery's "
                      f"streaming buffer (usually clears within a couple hours on its own). "
                      f"If this is test data and you don't want to wait, run reset_tables() "
                      f"(defined in Cell 1c) and re-upload.")
                return False
            raise
    return True


# %% [Cell 3] Stage 1: PDF -> per-page text extraction -> page-type classification -> bronze write
# Run `!pip install pymupdf` above if fitz isn't already available in this environment.
# NOTE: Gemini extraction and the silver-layer writes are deliberately NOT in this
# version — that's the next increment, once bronze output below looks right.
import fitz  # PyMuPDF

# First pass, built from titles observed in your two sample files. Anything that
# doesn't match falls through to 'unclassified' rather than being guessed at —
# check the printed output for those and add a pattern instead of assuming.
PAGE_TYPE_PATTERNS = [
    (r"Table of Contents", "toc"),
    (r"NOTICE OF CONFIDENTIALITY", "confidentiality_notice"),
    (r"COMPETITIVE INTELLIGENCE HEADLINES", "ci_headlines"),
    (r"CI Headlines", "section_divider"),  # short divider title, distinct from the phrase above
    (r"Major Offer Changes Effective", "major_offer_changes"),
    (r"Post-Paid Apple Smartphone Offers", "postpaid_device_offers_apple"),
    (r"Post-Paid Samsung and Google Smartphone Offers", "postpaid_device_offers_samsung_google"),
    (r"Post-Paid Motorola, Affordable Phones and BYOD", "postpaid_device_offers_motorola_affordable"),
    (r"Post-Paid BTS \(Watches\)", "postpaid_bts_offers"),
    (r"Post-Paid BTS \(Tablet\)", "postpaid_bts_offers"),
    (r"Post-Paid BTS & Other Offers", "postpaid_bts_offers"),
    (r"Post-Paid Service Offers", "postpaid_service_offers"),
    (r"Postpaid Cable MVNO Handset Offers", "cable_mvno_offers"),
    (r"Cable MVNO Service Pricing", "cable_mvno_offers"),
    (r"Business\s*[-–]\s*Flagship Device Offers", "business_device_offers"),
    (r"Key Highlights\s*[-–]\s*Week Ending", "national_retail_readout"),  # more reliable than the
    (r"Promotions Marketplace", "national_retail_readout"),               # Circana wordmark, which
                                                                            # may render as an image
    (r"Prepaid Brands\s*[-–]\s*Current Offers", "prepaid_brand_offers"),
    (r"Flagship Brands/Prepaid\s*[-–]\s*Current Offers", "prepaid_brand_offers"),
    (r"Walmart\s*[-–]\s*Current Offers", "prepaid_brand_offers"),
    (r"METRO BY T-MOBILE", "prepaid_price_grid_detail"),
    (r"Competitive Offers\s*&\s*Promotional\s*Updates", "section_divider"),
    (r"Competitive Offer Updates", "section_divider"),  # singular, no "& Promotional" — different divider
    (r"Business Internet Competitive Comparison", "business_internet_comparison"),
    (r"Competitive Comparison:\s*(TFB\s*)?Voice Plan", "voice_plan_comparison"),
    (r"Competitive Offer Report", "title_slide"),
    (r"ARE YOU WITH US", "closing_slide"),
]


# Patterns added here (via add_page_type_pattern, not code edits) are checked
# alongside PAGE_TYPE_PATTERNS above. This is what makes recognizing a new page
# type self-service: when a new report section shows up as 'unclassified', you
# can teach the system what it is from a notebook cell, without waiting on a
# code change. Cached per Stage 1 run — call load_custom_patterns(force_refresh=True)
# to pick up something added mid-session.
PAGE_TYPE_PATTERNS_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_control_pageTypePatterns"
# Note: no _weekly suffix — this isn't a weekly snapshot, it's a slowly-growing
# reference table, same reasoning as the offer-type taxonomy discussed earlier.

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{PAGE_TYPE_PATTERNS_TABLE}` (
        pattern STRING,
        page_type STRING,
        added_by STRING,
        added_at TIMESTAMP,
        notes STRING
    )
""").result()

_custom_patterns_cache = None


def load_custom_patterns(force_refresh=False):
    global _custom_patterns_cache
    if _custom_patterns_cache is not None and not force_refresh:
        return _custom_patterns_cache
    try:
        rows = bq.query(f"SELECT pattern, page_type FROM `{PAGE_TYPE_PATTERNS_TABLE}` ORDER BY added_at").result()
        _custom_patterns_cache = [(r["pattern"], r["page_type"]) for r in rows]
    except Exception:
        _custom_patterns_cache = []
    return _custom_patterns_cache


def add_page_type_pattern(pattern: str, page_type: str, notes: str = ""):
    """Call this after looking at an unclassified page (e.g. in the review table)
    to teach the system what it is — no code change, no waiting. Takes effect on
    the next Stage 1 run (classify_page checks this table alongside the built-in
    patterns)."""
    row = {
        "pattern": pattern, "page_type": page_type, "added_by": "khalid",
        "added_at": datetime.utcnow().isoformat(), "notes": notes,
    }
    write_row(PAGE_TYPE_PATTERNS_TABLE, row)
    load_custom_patterns(force_refresh=True)
    print(f"Added: {pattern!r} -> {page_type!r}. Applies starting with the next Stage 1 run.")
    print("Note: this only makes classify_page() recognize the page — it does NOT add "
          "extraction for it. That's still a separate decision (reuse an existing "
          "extractor if the shape matches, or build a new one if it doesn't).")


def suggest_page_type(page_text: str):
    """LLM-assisted starting point for categorizing an unclassified page — prints a
    suggestion, does NOT apply anything automatically. You still decide whether to
    call add_page_type_pattern() and what extraction (if any) it should get."""
    model = GenerativeModel(GEMINI_MODEL)
    prompt = f"""This is text from one page of a weekly telecom competitive
intelligence report. Suggest:
1. A short snake_case page_type label (e.g. "business_internet_comparison")
2. A one-sentence description of what the page contains
3. Which shape it resembles: a carrier-offer grid (carrier x device/item x offer,
   like the device/BTS/prepaid pages), a narrative/bulleted summary (like the CI
   headlines pages), a diff/changes table (introduced vs removed), or a numeric
   matrix with no carrier dimension (like a rate-card table) — this determines
   what extraction approach would actually fit.

PAGE TEXT:
{page_text[:3000]}"""
    response = model.generate_content(prompt)
    print(response.text)


def classify_page(text: str):
    normalized = re.sub(r"\s+", " ", text)  # collapse newlines/multi-space so line-wrap
                                              # differences between slide instances don't matter
    for pattern, page_type in PAGE_TYPE_PATTERNS + load_custom_patterns():
        if re.search(pattern, normalized, re.IGNORECASE):
            return page_type, pattern
    return "unclassified", None


# Pages with no offer content — the Gemini extraction stage (next increment) should
# skip these entirely rather than pay for a model call with nothing to extract.
# These are also exactly the pages with the duplicated-text rendering artifact
# (shadow-effect titles over photo backgrounds), so skipping them sidesteps that
# problem too rather than needing to solve it.
NON_CONTENT_PAGE_TYPES = {
    "title_slide", "confidentiality_notice", "toc", "section_divider", "closing_slide",
}


def reconstruct_layout_text(page, y_tolerance=3):
    """Cluster words into horizontal rows by y-position, sort left-to-right within
    each row — this replaces pypdf's layout mode, which produced empty table bodies
    on your files. y_tolerance is a starting guess: if printed rows below look merged
    or split wrong on a given page, that's the value to adjust."""
    words = page.get_text("words")  # (x0, y0, x1, y1, word, block, line, word_no)
    if not words:
        return ""
    words = sorted(words, key=lambda w: (round(w[1] / y_tolerance), w[0]))
    lines, current_bucket, current_line = [], None, []
    for w in words:
        bucket = round(w[1] / y_tolerance)
        if current_bucket is None or bucket == current_bucket:
            current_line.append(w)
            current_bucket = bucket
        else:
            lines.append(" ".join(x[4] for x in sorted(current_line, key=lambda x: x[0])))
            current_line, current_bucket = [w], bucket
    if current_line:
        lines.append(" ".join(x[4] for x in sorted(current_line, key=lambda x: x[0])))
    return "\n".join(lines)


KNOWN_COLUMN_HEADERS = {
    "t-mobile", "verizon", "at&t", "att", "xfinity", "spectrum", "comcast",
    "optimum", "metro", "boost", "cricket", "smartphones",
}


def find_header_columns(page, y_tolerance=3, min_words_in_row=3, min_gap=15):
    """Look for the column-header row in the top third of the page. Two signals,
    combined: (1) skip the very first row-cluster, since on every page checked so
    far that's the page title, not the header row underneath it; (2) among what's
    left, prefer rows containing actual known carrier names over rows that just
    happen to have the most spaced-out words — word count alone picked the title
    over the real header on page 10, which is what this is fixing."""
    words = page.get_text("words")
    if not words:
        return None
    page_height = page.rect.height
    candidates = [w for w in words if w[1] < page_height * 0.35]
    candidates = sorted(candidates, key=lambda w: (round(w[1] / y_tolerance), w[0]))
    rows = {}
    for w in candidates:
        bucket = round(w[1] / y_tolerance)
        rows.setdefault(bucket, []).append(w)

    sorted_buckets = sorted(rows.keys())
    if len(sorted_buckets) > 1:
        rows = {b: rows[b] for b in sorted_buckets[1:]}  # drop the title row

    scored = []
    for row_words in rows.values():
        row_words = sorted(row_words, key=lambda w: w[0])
        if len(row_words) < min_words_in_row:
            continue
        gaps = [row_words[i + 1][0] - row_words[i][2] for i in range(len(row_words) - 1)]
        if not gaps or min(gaps) <= min_gap:
            continue
        vocab_hits = sum(1 for w in row_words if w[4].lower().strip(":") in KNOWN_COLUMN_HEADERS)
        scored.append((vocab_hits, len(row_words), row_words))

    if not scored:
        return None
    scored.sort(key=lambda t: (t[0], t[1]), reverse=True)  # vocab matches win, word count as tiebreak
    return [w[0] for w in scored[0][2]]


def reconstruct_columns(page, y_tolerance=3):
    """Column-aware alternative to reconstruct_layout_text, for grid-table pages.
    Detects column boundaries from a header row, buckets every word into its
    column by x-position FIRST, then does row-clustering within each column
    separately. This avoids the cross-column bleeding seen when clustering by
    y-position across the full page width — different carriers wrapping to
    different numbers of lines for the same conceptual row was zippering
    unrelated columns' text together."""
    boundaries = find_header_columns(page, y_tolerance=y_tolerance)
    if not boundaries:
        return reconstruct_layout_text(page, y_tolerance=y_tolerance)  # fallback

    words = page.get_text("words")
    columns = {i: [] for i in range(len(boundaries))}
    for w in words:
        col_idx = 0
        for i, b in enumerate(boundaries):
            if w[0] >= b:
                col_idx = i
        columns[col_idx].append(w)

    blocks = []
    for i in range(len(boundaries)):
        col_words = sorted(columns[i], key=lambda w: (round(w[1] / y_tolerance), w[0]))
        lines, bucket, line = [], None, []
        for w in col_words:
            b = round(w[1] / y_tolerance)
            if bucket is None or b == bucket:
                line.append(w)
                bucket = b
            else:
                lines.append(" ".join(x[4] for x in sorted(line, key=lambda x: x[0])))
                line, bucket = [w], b
        if line:
            lines.append(" ".join(x[4] for x in sorted(line, key=lambda x: x[0])))
        blocks.append(f"[COLUMN {i + 1}]\n" + "\n".join(lines))
    return "\n\n".join(blocks)


def render_page_image(page, dpi=150) -> bytes:
    """Render a page to PNG bytes for the text+image Gemini stage. Only called for
    GRID_PAGE_TYPES pages — no point rendering (or storing) images for pages that
    are being handled text-only or skipped entirely."""
    pix = page.get_pixmap(dpi=dpi)
    return pix.tobytes("png")


def run_extraction_pipeline(run_id: str, report_date, pdf_bytes: bytes, file_name: str):
    load_custom_patterns(force_refresh=True)  # pick up anything added since the last run
    print(f"[{run_id}] Opening {file_name} ...")
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    print(f"[{run_id}] {len(doc)} pages found.\n")

    rows = []
    total_pages = len(doc)
    for i, page in enumerate(doc):
        page_num = i + 1
        layout_text = reconstruct_layout_text(page)
        page_type, matched_pattern = classify_page(layout_text)

        if page_type == "unclassified" and page_num == 1:
            # Same reasoning as the closing-slide fallback below: title slide text is
            # duplicated by what looks like a shadow-effect rendering (two overlapping
            # text boxes), which breaks literal-phrase matching unpredictably depending
            # on block order. Position is the reliable signal here, not the text.
            page_type, matched_pattern = "title_slide", "(fallback: first page of deck)"

        if page_type == "unclassified" and page_num == total_pages:
            # No text pattern is reliable here — the closing slide is a full-bleed
            # brand graphic with no real title text. Position relative to the deck's
            # own length (not an absolute page number) is safe to use as a fallback.
            page_type, matched_pattern = "closing_slide", "(fallback: last page of deck)"

        image_b64 = None
        stored_text = layout_text
        if page_type in GRID_PAGE_TYPES:
            image_b64 = base64.b64encode(render_page_image(page)).decode("ascii")
            # Store the column-aware reconstruction for grid pages, not the plain
            # row-based one — this is what actually feeds the Gemini prompt's
            # reference text and the hallucination guard downstream, so it should
            # be the more reliable version, not whatever happened to be computed
            # first for classification.
            stored_text = reconstruct_columns(page)

        print(f"--- Page {page_num} ---")
        print(f"  matched pattern : {matched_pattern!r}")
        print(f"  page_type       : {page_type}")
        print(f"  image rendered  : {image_b64 is not None}")
        print(f"  text preview    : {stored_text[:120]!r}")
        print()

        rows.append({
            "report_date": report_date.isoformat(),
            "run_id": run_id,
            "source_pdf": file_name,
            "page_number": page_num,
            "page_type": page_type,
            "text_extraction_raw": stored_text,
            "llm_extraction_raw": None,   # populated by the stage-2 extraction functions
            "page_image_png": image_b64,  # base64 string here; BigQuery decodes to BYTES on load
            "prompt_version": None,
            "extracted_at": datetime.utcnow().isoformat(),
        })

    unclassified = [r["page_number"] for r in rows if r["page_type"] == "unclassified"]
    if unclassified:
        print(f"[{run_id}] WARNING: {len(unclassified)} page(s) matched no known pattern: {unclassified}")
        print("These are still written to bronze as 'unclassified' — inspect them and add a pattern if they're real content pages, not just dividers/photos.\n")

    errors = None
    try:
        write_rows(BRONZE_TABLE, rows)
    except Exception as e:
        errors = e
    if errors:
        print(f"[{run_id}] BigQuery write failed: {errors}")
    else:
        print(f"[{run_id}] Wrote {len(rows)} rows to bronze.")


# %% [Cell 4] Upload widget + explicit Upload button + dedup check + decision UI

upload_widget = widgets.FileUpload(accept=".pdf", multiple=False, description="Choose file")
upload_button = widgets.Button(description="Upload", button_style="primary", icon="upload")
output = widgets.Output()
display(widgets.HBox([upload_widget, upload_button]), output)


def proceed(report_date, run_id, file_name, hash_, pdf_bytes, mode):
    with output:
        if mode == "cancel":
            print("Cancelled — nothing written.")
            return
        if mode == "replace":
            print(f"Deleting existing data for {report_date} ...")
            if not delete_existing_run(report_date):
                print("Aborting — old data wasn't fully cleared, so nothing new was written (avoids duplicates).")
                return
        write_manifest_row(report_date, run_id, file_name, hash_, status="active")
        run_extraction_pipeline(run_id, report_date, pdf_bytes, file_name)


_last_pdf_bytes = None
_last_file_name = None


def handle_upload(b):
    global _last_pdf_bytes, _last_file_name
    output.clear_output()
    if not upload_widget.value:
        with output:
            print("No file selected yet — choose a PDF first, then click Upload.")
        return

    uploaded = list(upload_widget.value.values())[0] if hasattr(upload_widget.value, "values") else upload_widget.value[0]
    file_name = uploaded["metadata"]["name"] if "metadata" in uploaded else uploaded.name
    pdf_bytes = uploaded["content"] if "content" in uploaded else uploaded.content
    _last_pdf_bytes, _last_file_name = pdf_bytes, file_name

    with output:
        report_date = parse_date_from_filename(file_name)
        if report_date is None:
            print(f"Couldn't parse a report date from filename '{file_name}' — check naming convention.")
            return

        match, found_date = verify_date_from_content(pdf_bytes, report_date)
        if match is False:
            print(f"Warning: filename implies {report_date}, but content suggests {found_date}. Proceeding with content date.")
            report_date = found_date

        hash_ = file_hash(pdf_bytes)
        existing = get_active_manifest_row(report_date)
        run_id = str(uuid.uuid4())

        if existing is None:
            print(f"No existing data for {report_date}. Proceeding.")
            proceed(report_date, run_id, file_name, hash_, pdf_bytes, mode="new")
            return

        print(f"Data for {report_date} already exists (run {existing['run_id']}, file '{existing['file_name']}').")
        print("Choose how to proceed:")

        replace_btn = widgets.Button(description="Replace", button_style="danger")
        append_btn = widgets.Button(description="Append", button_style="warning")
        cancel_btn = widgets.Button(description="Cancel", button_style="")
        display(widgets.HBox([replace_btn, append_btn, cancel_btn]))

        replace_btn.on_click(lambda b: proceed(report_date, run_id, file_name, hash_, pdf_bytes, "replace"))
        # NOTE on append: this leaves two 'active' manifest rows for the same report_date
        # (nothing gets deleted). Decide before relying on this: should gold then show
        # both runs, or should "append" in your workflow actually behave like replace
        # (e.g. if it's really just MCI sending a corrected re-issue of the same week)?
        append_btn.on_click(lambda b: proceed(report_date, run_id, file_name, hash_, pdf_bytes, "append"))
        cancel_btn.on_click(lambda b: proceed(report_date, run_id, file_name, hash_, pdf_bytes, "cancel"))


upload_button.on_click(handle_upload)


# %% [Cell 5] Diagnostic: full text reconstruction for one page, not just the 120-char
# preview. method="rows" is what's currently in the pipeline (Cell 3) — the one that
# just showed cross-column bleeding on page 10. method="columns" is the new
# column-aware alternative. Compare both on the same page before deciding which one
# actually goes into run_extraction_pipeline.
def inspect_page(page_number: int, method="columns", y_tolerance=3, pdf_bytes=None):
    pdf_bytes = pdf_bytes or _last_pdf_bytes
    if pdf_bytes is None:
        print("No PDF in memory yet — upload one first, or pass pdf_bytes explicitly.")
        return
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    page = doc[page_number - 1]
    text = reconstruct_columns(page, y_tolerance) if method == "columns" else reconstruct_layout_text(page, y_tolerance)
    print(text)

# inspect_page(10, method="columns")


# %% [Cell 6] Extraction-strategy tiers, and a full-deck dump for reviewing every
# page_type in one pass. GRID types get column-aware reconstruction (matches what
# they'll get: image + text in the Gemini stage). Everything else content-bearing
# gets plain row reconstruction (matches what they'll get: text-only). Non-content
# types are skipped, same as they will be in the real extraction stage.
GRID_PAGE_TYPES = {
    "postpaid_device_offers_apple",
    "postpaid_device_offers_samsung_google",
    "postpaid_device_offers_motorola_affordable",
    "postpaid_bts_offers",
    "postpaid_service_offers",
    "cable_mvno_offers",
    "business_device_offers",
    "prepaid_brand_offers",
    "prepaid_price_grid_detail",
    "major_offer_changes",
}


def dump_all_pages(pdf_bytes=None, y_tolerance=3):
    pdf_bytes = pdf_bytes or _last_pdf_bytes
    if pdf_bytes is None:
        print("No PDF in memory yet — upload one first.")
        return
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    total_pages = len(doc)

    for i, page in enumerate(doc):
        page_num = i + 1
        row_text = reconstruct_layout_text(page, y_tolerance)
        page_type, _ = classify_page(row_text)
        if page_type == "unclassified" and page_num == 1:
            page_type = "title_slide"
        if page_type == "unclassified" and page_num == total_pages:
            page_type = "closing_slide"

        strategy = (
            "skip" if page_type in NON_CONTENT_PAGE_TYPES
            else "text+image" if page_type in GRID_PAGE_TYPES
            else "text-only"
        )

        print(f"\n{'=' * 80}\nPAGE {page_num}  |  page_type: {page_type}  |  strategy: {strategy}\n{'=' * 80}")

        if strategy == "skip":
            print("(non-content page, no extraction planned)")
            continue

        print(reconstruct_columns(page, y_tolerance) if strategy == "text+image" else row_text)

# dump_all_pages()


# %% [Cell 7] Stage 2, text-only tier: Gemini extraction for ci_headlines and
# national_retail_readout. Reads from bronze (not the PDF directly), so this can
# be re-run against an existing run_id without re-uploading or re-parsing.

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{SILVER_CI_HEADLINES_TABLE}` (
        report_date DATE,
        run_id STRING,
        source_page INT64,
        carrier STRING,
        section STRING,
        headline_title STRING,
        headline_detail STRING,
        effective_date_range STRING,
        has_changes BOOL,
        values_agree BOOL,
        llm_extraction_raw STRING,
        extracted_at TIMESTAMP
    )
""").result()

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{SILVER_RETAIL_TABLE}` (
        report_date DATE,
        run_id STRING,
        source_page INT64,
        week_ending STRING,
        retailer STRING,
        device STRING,
        discount_amount STRING,
        final_price STRING,
        carrier_service STRING,
        note STRING,
        values_agree BOOL,
        llm_extraction_raw STRING,
        extracted_at TIMESTAMP
    )
""").result()

print("Text-only silver tables ready.")


CI_HEADLINES_SCHEMA = {
    "type": "object",
    "properties": {
        "headlines": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "carrier": {"type": "string"},
                    "section": {"type": "string"},
                    "headline_title": {"type": "string"},
                    "headline_detail": {"type": "string"},
                    "effective_date_range": {"type": "string"},
                    "has_changes": {"type": "boolean"},
                },
                "required": ["carrier", "headline_title", "has_changes"],
            },
        }
    },
    "required": ["headlines"],
}

CI_HEADLINES_PROMPT = """You are extracting structured data from a competitive \
intelligence report page. The page contains a numbered list of carriers, each \
followed by either specific bulleted headline items describing changes, or a \
simple "No changes" note.

For EACH numbered carrier entry:
- If it says "No changes" (or equivalent), emit exactly one record with \
has_changes=false, headline_title="No changes", headline_detail="".
- Otherwise, emit one record per bullet point, with has_changes=true.

Extract "section" from the page's subheading (e.g. "Postpaid & Cable MVNO Key \
Highlights"). Extract any date or date range in parentheses into \
effective_date_range if present, otherwise leave it blank. Do not paraphrase or \
summarize headline_detail — keep it close to the source wording, since this \
feeds a business report.

PAGE TEXT:
{page_text}"""


RETAIL_READOUT_SCHEMA = {
    "type": "object",
    "properties": {
        "week_ending": {"type": "string"},
        "retail_offers": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "retailer": {"type": "string"},
                    "device": {"type": "string"},
                    "discount_amount": {"type": "string"},
                    "final_price": {"type": "string"},
                    "carrier_service": {"type": "string"},
                    "note": {"type": "string"},
                },
                "required": ["retailer", "device"],
            },
        },
    },
    "required": ["retail_offers"],
}

RETAIL_READOUT_PROMPT = """You are extracting structured retail promotion data \
from a weekly competitive intelligence report page. The page is prose-style \
bullets describing which retailers are running which device discounts.

Extract "week_ending" from the page header. For each distinct retailer+device \
offer mentioned, extract one record with: retailer, device (model name only), \
discount_amount (as a dollar string, e.g. "$350"), final_price if mentioned, \
carrier_service if the offer is tied to a specific carrier/MVNO (otherwise \
"unlocked"), and a short note for any other relevant detail (e.g. comparison to \
last week). If a bullet mentions no specific offer (e.g. "no new promotions"), \
skip it — don't emit an empty record.

PAGE TEXT:
{page_text}"""


def call_gemini_json(prompt: str, schema: dict):
    model = GenerativeModel(GEMINI_MODEL)
    response = model.generate_content(
        prompt,
        generation_config=GenerationConfig(
            response_mime_type="application/json",
            response_schema=schema,
            temperature=0,  # minimize drift on what should be a factual extraction
            max_output_tokens=32768,  # raised after page 20/21's dense multi-tier
                                       # fan-out truncated mid-response with the default
        ),
    )
    return parse_json_with_recovery(response.text)


def parse_json_with_recovery(text: str) -> dict:
    """A truncated response (hit the token limit mid-generation, even at 32768)
    should lose only the last partial record, not the entire page — trim back to
    the last complete array entry and close it out rather than raising."""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        last_complete = text.rfind("},")
        if last_complete == -1:
            raise
        salvaged = text[:last_complete + 1] + "]}"
        result = json.loads(salvaged)  # let this raise if still broken — don't mask a real failure
        print(f"    (response was truncated — recovered {len(result.get('offers', []))} record(s) before the cutoff)")
        return result


def extract_numeric_value(text: str):
    """Pull a normalized number out of a dollar-figure string, handling 'k'
    abbreviations (1.1k -> 1100) and commas, so '$1,100', '$1.1k', and '1100'
    all compare equal."""
    if not text:
        return None
    m = re.search(r"([\d,]+\.?\d*)\s*(k)?", text, re.IGNORECASE)
    if not m:
        return None
    try:
        num = float(m.group(1).replace(",", ""))
    except ValueError:
        return None
    return num * 1000 if m.group(2) else num


def values_present_in_text(record: dict, keys: list, source_text: str) -> bool:
    """Cheap hallucination guard — checks that key extracted values are actually
    grounded in the bronze text for this page. Layered, in order of strictness:
    1. Literal substring match (whitespace-normalized) — same as before.
    2. Strikethrough collapse: source often shows 'OLD NEW' price pairs back to
       back (an old price struck through, replaced by the current one right
       next to it) — seen recurring across multiple weeks and page types. The
       extracted value is correctly just the current price, which won't appear
       as a contiguous substring with the old one sitting in between.
    3. Numeric fallback: if the literal string still isn't found, check whether
       ANY dollar figure in the source matches numerically once 'k' abbreviations
       and a leading '<' are normalized away (e.g. source '<$1.1k' vs extracted
       '$1100', or source '$50' vs extracted '$50/mo'). This is deliberately
       looser — it trades some precision (a coincidentally-matching number
       elsewhere on the page could slip through) for not flagging correct,
       usefully-normalized extractions as if they were hallucinations. Worth
       revisiting if it turns out to let real errors through."""
    normalized_source = re.sub(r"\s+", " ", source_text)
    collapsed_source = re.sub(r"\$[\d,]+\.?\d*\s+(?=\$)", "", normalized_source)
    source_numbers = {extract_numeric_value(m) for m in
                       re.findall(r"[<>]?\$[\d,]+\.?\d*k?", normalized_source, re.IGNORECASE)}

    for key in keys:
        val = str(record.get(key, "")).strip()
        if not val:
            continue
        val_norm = re.sub(r"\s+", " ", val)
        if val_norm in collapsed_source:
            continue
        target_num = extract_numeric_value(val)
        if target_num is not None and target_num in source_numbers:
            continue
        return False
    return True


def print_extraction_results(rows: list, source_text: str, sample_ok: int = 3):
    """Consistent logging for extraction results: full detail (with inline source
    context) on everything flagged, a small representative sample of what looks
    fine, and a count summary — pasting this back should be enough to diagnose
    without a separate debug_amount_mismatches call every time."""
    ok_rows = [r for r in rows if r["values_agree"]]
    check_rows = [r for r in rows if not r["values_agree"]]
    print(f"  {len(rows)} record(s): {len(ok_rows)} OK, {len(check_rows)} flagged")

    if check_rows:
        normalized_source = re.sub(r"\s+", " ", source_text)
        print(f"  --- flagged, full detail ---")
        for r in check_rows:
            print(f"    [CHECK] {r['carrier']}: {r['item_name']} — {r['offer_amount']!r}"
                  + (f"  (tier: {r['plan_tier']})" if r.get("plan_tier") else ""))
            # Prefer locating context by the actual number — carrier names repeat
            # often enough on dense pages that .find() on the carrier alone tends
            # to land on an unrelated header mention rather than the real bullet.
            num_match = re.search(r"[\d,]+", r["offer_amount"] or "")
            idx = normalized_source.find(num_match.group()) if num_match else -1
            if idx == -1 and r.get("carrier"):
                idx = normalized_source.find(r["carrier"])
            if idx == -1 and r.get("carrier"):
                idx = normalized_source.find(r["carrier"].split()[0])  # e.g. "Cricket" vs "Cricket Wireless"
            if idx >= 0:
                print(f"       nearby source text: ...{normalized_source[max(0, idx - 40):idx + 150]}...")
            else:
                print(f"       (nothing matching found in source text at all)")

    if ok_rows:
        shown = ok_rows[:sample_ok]
        print(f"  --- sample of OK rows ({len(shown)} of {len(ok_rows)}) ---")
        for r in shown:
            print(f"    [OK] {r['carrier']}: {r['item_name']} — {r['offer_amount']}")


def print_flag_breakdown(review_rows: list):
    """Run-level summary of *why* things got flagged, so the top-line signal is
    visible without scrolling back through every page's detail."""
    if not review_rows:
        return
    counts = {}
    for r in review_rows:
        counts[r["flag_reason"]] = counts.get(r["flag_reason"], 0) + 1
    print("  Breakdown:", ", ".join(f"{reason}={n}" for reason, n in sorted(counts.items())))


def fetch_bronze_pages(run_id, page_types):
    query = f"""
        SELECT page_number, page_type, text_extraction_raw
        FROM `{BRONZE_TABLE}`
        WHERE run_id = @run_id AND page_type IN UNNEST(@page_types)
        ORDER BY page_number
    """
    job = bq.query(
        query,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter("run_id", "STRING", run_id),
                bigquery.ArrayQueryParameter("page_types", "STRING", page_types),
            ]
        ),
    )
    return list(job.result())


def extract_ci_headlines(page_number, page_text, report_date, run_id):
    result = call_gemini_json(CI_HEADLINES_PROMPT.format(page_text=page_text), CI_HEADLINES_SCHEMA)
    rows = []
    for h in result.get("headlines", []):
        rows.append({
            "report_date": report_date.isoformat(),
            "run_id": run_id,
            "source_page": page_number,
            "carrier": h.get("carrier"),
            "section": h.get("section"),
            "headline_title": h.get("headline_title"),
            "headline_detail": h.get("headline_detail"),
            "effective_date_range": h.get("effective_date_range"),
            "has_changes": h.get("has_changes"),
            "values_agree": values_present_in_text(h, ["carrier"], page_text),
            "llm_extraction_raw": json.dumps(result),
            "extracted_at": datetime.utcnow().isoformat(),
        })
    return rows


def extract_national_retail(page_number, page_text, report_date, run_id):
    result = call_gemini_json(RETAIL_READOUT_PROMPT.format(page_text=page_text), RETAIL_READOUT_SCHEMA)
    week_ending = result.get("week_ending")
    rows = []
    for o in result.get("retail_offers", []):
        rows.append({
            "report_date": report_date.isoformat(),
            "run_id": run_id,
            "source_page": page_number,
            "week_ending": week_ending,
            "retailer": o.get("retailer"),
            "device": o.get("device"),
            "discount_amount": o.get("discount_amount"),
            "final_price": o.get("final_price"),
            "carrier_service": o.get("carrier_service"),
            "note": o.get("note"),
            "values_agree": values_present_in_text(o, ["retailer", "device", "discount_amount"], page_text),
            "llm_extraction_raw": json.dumps(result),
            "extracted_at": datetime.utcnow().isoformat(),
        })
    return rows


def run_text_only_extraction(run_id, report_date):
    print(f"[{run_id}] Text-only extraction stage starting...")
    ci_rows, retail_rows = [], []

    ci_pages = fetch_bronze_pages(run_id, ["ci_headlines"])
    retail_pages = fetch_bronze_pages(run_id, ["national_retail_readout"])
    if not ci_pages and not retail_pages:
        print(f"  No matching bronze pages found for run_id {run_id!r} — check show_bronze_state() "
              f"for the actual current run_id (this commonly happens from pasting a placeholder "
              f"value like \"...\" instead of a real run_id).")
        return

    for page in ci_pages:
        print(f"\n--- Page {page['page_number']} (ci_headlines) ---")
        try:
            rows = extract_ci_headlines(page["page_number"], page["text_extraction_raw"], report_date, run_id)
            print(f"  extracted {len(rows)} headline records")
            for r in rows:
                flag = "OK   " if r["values_agree"] else "CHECK"
                print(f"    [{flag}] {r['carrier']}: {r['headline_title']}")
            ci_rows.extend(rows)
        except Exception as e:
            print(f"  FAILED: {e}")

    for page in retail_pages:
        print(f"\n--- Page {page['page_number']} (national_retail_readout) ---")
        try:
            rows = extract_national_retail(page["page_number"], page["text_extraction_raw"], report_date, run_id)
            print(f"  extracted {len(rows)} retail offer records")
            for r in rows:
                flag = "OK   " if r["values_agree"] else "CHECK"
                print(f"    [{flag}] {r['retailer']}: {r['device']} ({r['discount_amount']})")
            retail_rows.extend(rows)
        except Exception as e:
            print(f"  FAILED: {e}")

    if ci_rows:
        delete_silver_rows_for_date(SILVER_CI_HEADLINES_TABLE, report_date)
        write_rows(SILVER_CI_HEADLINES_TABLE, ci_rows)
        print(f"\n[{run_id}] Wrote {len(ci_rows)} rows to silver ci_headlines (prior rows for this date cleared first).")
    if retail_rows:
        delete_silver_rows_for_date(SILVER_RETAIL_TABLE, report_date)
        write_rows(SILVER_RETAIL_TABLE, retail_rows)
        print(f"[{run_id}] Wrote {len(retail_rows)} rows to silver national_retail_readout (prior rows for this date cleared first).")

# run_text_only_extraction(run_id="...", report_date=date(2026, 5, 29))


# %% [Cell 8] Diagnostic: what's actually in bronze right now, and does the
# run_id you're using match it. Run this before re-trying run_text_only_extraction.
def show_bronze_state():
    print(f"[queried at {datetime.utcnow().isoformat()} UTC]\n")
    query = f"""
        SELECT run_id, report_date, page_type, COUNT(*) AS page_count
        FROM `{BRONZE_TABLE}`
        GROUP BY run_id, report_date, page_type
        ORDER BY report_date DESC, run_id, page_type
    """
    for row in bq.query(query).result():
        print(f"{row['report_date']}  |  run_id {row['run_id']}  |  {row['page_type']:35s}  |  {row['page_count']} page(s)")

show_bronze_state()


# %% [Cell 8b] Diagnostic: for any page with values_agree=False rows, show exactly
# what text those rows were checked against — direct inspection instead of guessing
# why a substring match failed.
def debug_amount_mismatches(run_id, page_number):
    rows = list(bq.query(
        f"""SELECT carrier, item_name, plan_tier, offer_amount
            FROM `{SILVER_DEVICE_OFFERS_TABLE}`
            WHERE run_id = @run_id AND source_page = @page_number AND values_agree = FALSE""",
        job_config=bigquery.QueryJobConfig(query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", run_id),
            bigquery.ScalarQueryParameter("page_number", "INT64", page_number),
        ]),
    ).result())

    bronze = list(bq.query(
        f"""SELECT text_extraction_raw FROM `{BRONZE_TABLE}`
            WHERE run_id = @run_id AND page_number = @page_number""",
        job_config=bigquery.QueryJobConfig(query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", run_id),
            bigquery.ScalarQueryParameter("page_number", "INT64", page_number),
        ]),
    ).result())
    normalized = re.sub(r"\s+", " ", bronze[0]["text_extraction_raw"]) if bronze else ""

    for r in rows:
        amt = r["offer_amount"] or ""
        found = amt in normalized
        print(f"{r['carrier']:10s} {r['item_name']:20s} tier={r['plan_tier'] or '-':8s} amount={amt!r:35s} literal_match={found}")
        if not found and r["carrier"]:
            idx = normalized.find(r["carrier"])
            print(f"   nearby context: ...{normalized[max(0, idx):idx + 250]}...")

# debug_amount_mismatches("66d0eebf-ee3f-452f-97a2-b2406625f177", 10)


# %% [Cell 9] Stage 2, text+image tier — FIRST GRID GROUP ONLY: postpaid device
# offers (postpaid_device_offers_apple/samsung_google/motorola_affordable, i.e.
# pages 10-12). Deliberately scoped to just this one group before wiring up the
# other six grid page types — same reasoning as starting with page 10 alone
# instead of assuming the fix would generalize.

SILVER_DEVICE_OFFERS_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_silver_postpaidDeviceOffers_weekly"

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{SILVER_DEVICE_OFFERS_TABLE}` (
        report_date DATE,
        run_id STRING,
        source_page INT64,
        device_category STRING,   -- apple | samsung_google | motorola_affordable
        carrier STRING,
        item_name STRING,
        device_group STRING,      -- comma-separated devices that shared this offer, incl. item_name
        plan_tier STRING,         -- e.g. Good/Better/Best when pricing varies by rate plan tier
        offer_type STRING,
        offer_amount STRING,
        offer_condition STRING,
        values_agree BOOL,
        llm_extraction_raw STRING,
        extracted_at TIMESTAMP
    )
""").result()

print("Silver postpaidDeviceOffers table ready.")

bq.query(f"ALTER TABLE `{SILVER_DEVICE_OFFERS_TABLE}` ADD COLUMN IF NOT EXISTS device_group STRING").result()
bq.query(f"ALTER TABLE `{SILVER_DEVICE_OFFERS_TABLE}` ADD COLUMN IF NOT EXISTS plan_tier STRING").result()


# Review table — one consolidated queue, not per-section, covering the three
# things you flagged: page classification failures, unrecognized carriers, and
# value mismatches. flag_reason distinguishes which.
REVIEW_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_review_flaggedOffers_weekly"

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{REVIEW_TABLE}` (
        report_date DATE,
        run_id STRING,
        source_page INT64,
        source_table STRING,
        flag_reason STRING,     -- 'unclassified_page' | 'extraction_failed' | 'new_carrier' | 'amount_mismatch'
        carrier STRING,
        item_name STRING,
        offer_amount STRING,
        detail STRING,
        status STRING,          -- 'pending' | 'resolved'
        flagged_at TIMESTAMP
    )
""").result()

print("Review table ready.")


# Carrier list — hardcoded rather than a live view. Carriers are a small, stable
# set (same handful across every page type seen so far), and this overlaps almost
# entirely with KNOWN_COLUMN_HEADERS used for column detection. A live view added
# a BigQuery round-trip and an empty-on-first-run edge case for no real benefit —
# extend this list by hand on the rare occasion a genuinely new carrier shows up.
KNOWN_CARRIERS = [
    "T-Mobile", "Verizon", "AT&T", "Xfinity", "Spectrum", "Comcast",
    "Optimum", "Metro by T-Mobile", "Boost Mobile", "Cricket Wireless",
    "Straight Talk", "Total Wireless", "T-Mobile Prepaid", "AT&T Prepaid",
    "Verizon Prepaid",
]


def get_known_carriers():
    return KNOWN_CARRIERS


def write_review_rows(rows: list):
    if rows:
        write_rows(REVIEW_TABLE, rows)


def flag_unclassified_pages(run_id, report_date):
    """Covers 'is classification working' — checked separately from the Gemini
    extraction stage, since an unclassified page never reaches Gemini at all."""
    pages = fetch_bronze_pages(run_id, ["unclassified"])
    if not pages:
        return
    rows = [{
        "report_date": report_date.isoformat(), "run_id": run_id, "source_page": p["page_number"],
        "source_table": "bronze", "flag_reason": "unclassified_page", "carrier": None,
        "item_name": None, "offer_amount": None,
        "detail": f"No page_type pattern matched: {p['text_extraction_raw'][:150]!r}",
        "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
    } for p in pages]
    write_review_rows(rows)
    print(f"  Flagged {len(rows)} unclassified page(s) for review.")


def get_known_page_types():
    """Union of every page_type the pipeline currently recognizes, built
    dynamically so this stays in sync as patterns get added — nothing to
    maintain by hand."""
    types = {pt for _, pt in PAGE_TYPE_PATTERNS} | {pt for _, pt in load_custom_patterns()} | NON_CONTENT_PAGE_TYPES
    return sorted(types)


PAGE_TYPE_CLASSIFY_SCHEMA = {
    "type": "object",
    "properties": {
        "page_type": {"type": "string"},
        "is_new": {"type": "boolean"},
        "distinctive_phrase": {"type": "string"},
        "reasoning": {"type": "string"},
    },
    "required": ["page_type", "is_new", "distinctive_phrase"],
}

AUTO_CLASSIFY_PROMPT = """You are classifying a page from a recurring weekly telecom \
competitive intelligence report. Page types already handled by this pipeline: \
{known_types}

Look at the page text below and decide:
- If it matches one of the known types above (just worded differently than whatever \
literal phrase currently catches that type), return that exact page_type and \
is_new=false.
- If it's genuinely a new kind of content not covered by any known type, propose a \
short new snake_case page_type name and set is_new=true.

Either way, also return distinctive_phrase: a short literal phrase (5-10 words) near \
the top of this page that would reliably identify this same page type in future \
weekly reports — this becomes a regex pattern, so pick a section title or fixed \
label, not a date or number that will change week to week.

PAGE TEXT:
{page_text}"""


def auto_classify_unclassified_pages(run_id, report_date):
    """Runs an LLM classification pass over anything Stage 1's regex patterns
    missed. Matching an existing type is applied immediately (low risk — it's a
    routing label, not a data value). A genuinely new type is also auto-registered
    (so it's not stuck in limbo), but stays visible via a review flag rather than
    disappearing silently — same principle as the carrier taxonomy, lighter touch
    since the stakes here are lower."""
    pages = fetch_bronze_pages(run_id, ["unclassified"])
    if not pages:
        return
    print(f"  Auto-classifying {len(pages)} unclassified page(s)...")
    known_types = get_known_page_types()
    still_unclassified = []

    for page in pages:
        try:
            prompt = AUTO_CLASSIFY_PROMPT.format(
                known_types=", ".join(known_types),
                page_text=page["text_extraction_raw"][:3000],
            )
            result = call_gemini_json(prompt, PAGE_TYPE_CLASSIFY_SCHEMA)
            page_type = result["page_type"]
            phrase = result["distinctive_phrase"]
            is_new = result["is_new"]

            bq.query(
                f"UPDATE `{BRONZE_TABLE}` SET page_type = @page_type WHERE run_id = @run_id AND page_number = @page_number",
                job_config=bigquery.QueryJobConfig(query_parameters=[
                    bigquery.ScalarQueryParameter("page_type", "STRING", page_type),
                    bigquery.ScalarQueryParameter("run_id", "STRING", run_id),
                    bigquery.ScalarQueryParameter("page_number", "INT64", page["page_number"]),
                ]),
            ).result()
            add_page_type_pattern(re.escape(phrase), page_type,
                                   notes=f"auto-added, run {run_id}, page {page['page_number']}")

            print(f"    Page {page['page_number']}: {'NEW category' if is_new else 'matched existing'} "
                  f"-> page_type={page_type!r}  (pattern: {phrase!r})")

            if is_new:
                write_review_rows([{
                    "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page["page_number"],
                    "source_table": "bronze", "flag_reason": "new_page_type_auto_created",
                    "carrier": None, "item_name": None, "offer_amount": None,
                    "detail": f"Auto-created page_type '{page_type}' — no extraction built for it yet. "
                              f"Reasoning: {result.get('reasoning', '')}",
                    "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
                }])
        except Exception as e:
            print(f"    Page {page['page_number']}: classification FAILED ({e}) — falling back to plain flag")
            still_unclassified.append(page)

    if still_unclassified:
        rows = [{
            "report_date": report_date.isoformat(), "run_id": run_id, "source_page": p["page_number"],
            "source_table": "bronze", "flag_reason": "unclassified_page", "carrier": None,
            "item_name": None, "offer_amount": None,
            "detail": f"No page_type pattern matched, and auto-classification failed: {p['text_extraction_raw'][:150]!r}",
            "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
        } for p in still_unclassified]
        write_review_rows(rows)
        print(f"  Flagged {len(rows)} page(s) that auto-classification couldn't handle.")


GRID_OFFERS_SCHEMA = {
    "type": "object",
    "properties": {
        "offers": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "carrier": {"type": "string"},
                    "item_name": {"type": "string"},
                    "device_group": {"type": "string"},
                    "plan_tier": {"type": "string"},
                    "offer_type": {"type": "string"},
                    "offer_amount": {"type": "string"},
                    "offer_condition": {"type": "string"},
                },
                "required": ["carrier", "item_name", "device_group"],
            },
        }
    },
    "required": ["offers"],
}

GRID_OFFERS_PROMPT = """You are extracting structured offer data from a table on \
this page. The IMAGE shows the actual visual table layout — use it to correctly \
associate each carrier's column with the right device row. Carriers often wrap \
to different numbers of lines for the same conceptual row, so column position in \
the image matters more than any text ordering.

The text below is a secondary, position-imperfect reference from the same page — \
use it to confirm exact dollar figures and wording, not to determine which \
carrier an offer belongs to; trust the image for that.

IMPORTANT — two ways a single line item can actually represent multiple offers, \
handle both by emitting separate records rather than combining values:
1. One offer applies to several devices at once (e.g. bullets under "All Apple \
Flagships" spanning iPhone 17, iPhone Air, iPhone 17 Pro, iPhone 17 Pro Max) — \
emit ONE RECORD PER DEVICE, repeating the shared values, and set device_group to \
the full comma-separated list of devices that share it (including the device \
itself). For a standalone offer, device_group should just equal item_name.
2. One device has different prices per rate plan tier (e.g. "$0/mo. Good | \
$5.55/mo. Better | $11.11/mo. Best") — emit ONE RECORD PER TIER, with plan_tier \
set to that tier's name (e.g. "Good") and offer_amount set to only that tier's \
price. Do NOT combine multiple tier prices into one offer_amount string. If \
pricing doesn't vary by tier, leave plan_tier blank.

Known carriers from prior reports — use these exact names when the offer is \
from one of them, for naming consistency across weeks: {known_carriers}

For each carrier + device (+ tier, if applicable) combination with a stated \
offer, extract one record: carrier, item_name (exactly one device name, e.g. \
"iPhone 17"), device_group, plan_tier (if applicable, else blank), offer_type \
(your best label: discount, bill credit, trade-in credit, BOGO, etc.), \
offer_amount (dollar figure for exactly this one record, as it appears), \
offer_condition (the requirement text, e.g. "w/ AAL & Any UNL plan"). Skip \
entries with no stated offer ("No offer", "--", "N/A").

REFERENCE TEXT:
{page_text}"""


def call_gemini_json_with_image(prompt: str, schema: dict, image_bytes: bytes):
    from vertexai.generative_models import Part
    model = GenerativeModel(GEMINI_MODEL)
    image_part = Part.from_data(data=image_bytes, mime_type="image/png")
    response = model.generate_content(
        [image_part, prompt],
        generation_config=GenerationConfig(
            response_mime_type="application/json",
            response_schema=schema,
            temperature=0,
            max_output_tokens=32768,
        ),
    )
    return parse_json_with_recovery(response.text)


def delete_silver_rows_for_date(table_id, report_date):
    """Silver tables never got the replace discipline bronze has — every re-run
    while iterating on extraction logic just appended on top of prior attempts.
    Call this before writing so a fresh extraction fully replaces old ones for
    the same report_date, regardless of which run_id produced them."""
    bq.query(
        f"DELETE FROM `{table_id}` WHERE report_date = @report_date",
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("report_date", "DATE", report_date)]
        ),
    ).result()


DEVICE_CATEGORY_MAP = {
    "postpaid_device_offers_apple": "apple",
    "postpaid_device_offers_samsung_google": "samsung_google",
    "postpaid_device_offers_motorola_affordable": "motorola_affordable",
}


def fetch_bronze_pages_with_image(run_id, page_types):
    query = f"""
        SELECT page_number, page_type, text_extraction_raw, page_image_png
        FROM `{BRONZE_TABLE}`
        WHERE run_id = @run_id AND page_type IN UNNEST(@page_types)
        ORDER BY page_number
    """
    job = bq.query(
        query,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter("run_id", "STRING", run_id),
                bigquery.ArrayQueryParameter("page_types", "STRING", page_types),
            ]
        ),
    )
    return list(job.result())


def extract_device_offers(page_number, page_type, page_text, image_bytes, report_date, run_id, known_carriers):
    prompt = GRID_OFFERS_PROMPT.format(
        page_text=page_text,
        known_carriers=", ".join(known_carriers) if known_carriers else "(none yet — first run)",
    )
    result = call_gemini_json_with_image(prompt, GRID_OFFERS_SCHEMA, image_bytes)
    offers = result.get("offers", [])

    # De-dupe exact-repeat records within the model's own response array — seen
    # on dense pages (same block of records repeated 2-3x in one response),
    # likely a generation-length artifact on long structured outputs rather than
    # anything about the prompt/schema itself. Key is deliberately narrow:
    # device_group and offer_condition are free-text fields the model can
    # rephrase slightly between repeats of "the same" row (extra word, different
    # comma placement), which let real duplicates slip past a stricter key that
    # included them — carrier + item_name + plan_tier + offer_amount is what
    # actually determines whether two rows represent the same real offer.
    seen, deduped_offers = set(), []
    for o in offers:
        sig = (o.get("carrier"), o.get("item_name"), o.get("plan_tier"), o.get("offer_amount"))
        if sig not in seen:
            seen.add(sig)
            deduped_offers.append(o)
    if len(deduped_offers) < len(offers):
        print(f"  (removed {len(offers) - len(deduped_offers)} exact-duplicate record(s) from model response)")

    rows, review_rows = [], []
    for o in deduped_offers:
        carrier = o.get("carrier")
        # Narrowed from checking item_name too, per what page 10 showed: multi-device
        # phrasing legitimately won't appear verbatim in source text even when correct,
        # so item_name was producing false positives. offer_amount is the field where
        # a mismatch actually matters.
        amount_ok = values_present_in_text(o, ["offer_amount"], page_text)
        rows.append({
            "report_date": report_date.isoformat(),
            "run_id": run_id,
            "source_page": page_number,
            "device_category": DEVICE_CATEGORY_MAP.get(page_type, page_type),
            "carrier": carrier,
            "item_name": o.get("item_name"),
            "device_group": o.get("device_group"),
            "plan_tier": o.get("plan_tier"),
            "offer_type": o.get("offer_type"),
            "offer_amount": o.get("offer_amount"),
            "offer_condition": o.get("offer_condition"),
            "values_agree": amount_ok,
            "llm_extraction_raw": json.dumps(result),
            "extracted_at": datetime.utcnow().isoformat(),
        })

        if carrier and known_carriers and carrier not in known_carriers:
            review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
                "source_table": "silver_postpaidDeviceOffers", "flag_reason": "new_carrier",
                "carrier": carrier, "item_name": o.get("item_name"), "offer_amount": o.get("offer_amount"),
                "detail": f"'{carrier}' not in known carrier list — new carrier, or naming inconsistency?",
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })
        elif not amount_ok:
            review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
                "source_table": "silver_postpaidDeviceOffers", "flag_reason": "amount_mismatch",
                "carrier": carrier, "item_name": o.get("item_name"), "offer_amount": o.get("offer_amount"),
                "detail": "offer_amount doesn't appear verbatim in the page's extracted text.",
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })

    return rows, review_rows


def run_device_offers_extraction(run_id, report_date):
    print(f"[{run_id}] Device offers (text+image) extraction starting...")

    auto_classify_unclassified_pages(run_id, report_date)

    known_carriers = get_known_carriers()
    print(f"  Grounding with {len(known_carriers)} known carrier(s): {known_carriers}")

    all_rows, all_review_rows = [], []
    pages = fetch_bronze_pages_with_image(run_id, list(DEVICE_CATEGORY_MAP.keys()))
    if not pages:
        print("  No matching bronze pages found for this run_id — check show_bronze_state().")
        return

    for page in pages:
        print(f"\n--- Page {page['page_number']} ({page['page_type']}) ---")
        if not page["page_image_png"]:
            print("  SKIPPED: no image stored for this page (was Stage 1 re-run after the image-rendering change?)")
            continue
        try:
            rows, review_rows = extract_device_offers(
                page["page_number"], page["page_type"], page["text_extraction_raw"],
                page["page_image_png"], report_date, run_id, known_carriers,
            )
            print(f"  extracted {len(rows)} offer records ({len(review_rows)} flagged for review)")
            print_extraction_results(rows, page["text_extraction_raw"])
            all_rows.extend(rows)
            all_review_rows.extend(review_rows)
        except Exception as e:
            print(f"  FAILED: {e}")
            all_review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page["page_number"],
                "source_table": "silver_postpaidDeviceOffers", "flag_reason": "extraction_failed",
                "carrier": None, "item_name": None, "offer_amount": None, "detail": str(e),
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })

    if all_rows:
        delete_silver_rows_for_date(SILVER_DEVICE_OFFERS_TABLE, report_date)
        write_rows(SILVER_DEVICE_OFFERS_TABLE, all_rows)
        print(f"\n[{run_id}] Wrote {len(all_rows)} rows to silver postpaidDeviceOffers (prior rows for this date cleared first).")
    if all_review_rows:
        write_review_rows(all_review_rows)
        print(f"[{run_id}] Flagged {len(all_review_rows)} row(s) for review.")
        print_flag_breakdown(all_review_rows)

# run_device_offers_extraction(run_id="...", report_date=date(2026, 5, 29))


# %% [Cell 10] Stage 2, TEXT-FIRST tier: covers the four remaining carrier-grid page
# types that share the same carrier x item x offer shape as device offers, but
# haven't had any extraction built yet — postpaid_bts_offers, cable_mvno_offers,
# business_device_offers, prepaid_brand_offers. Text-only by default; bronze
# already has images stored for these pages (from GRID_PAGE_TYPES rendering), so
# escalating a specific page_type to text+image later is a one-line change, not
# a re-run of Stage 1. major_offer_changes and prepaid_price_grid_detail are
# deliberately NOT included here — different shapes (diff-table, numeric matrix
# with no carrier dimension), not worth forcing into this schema.

SILVER_GRID_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_silver_carrierOfferGrid_weekly"

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{SILVER_GRID_TABLE}` (
        report_date DATE,
        run_id STRING,
        source_page INT64,
        page_type STRING,         -- which section this came from
        carrier STRING,
        item_name STRING,
        item_group STRING,        -- other items that shared this exact offer (like device_group)
        plan_tier STRING,
        offer_type STRING,
        offer_amount STRING,
        offer_condition STRING,
        values_agree BOOL,
        extraction_method STRING, -- 'text' or 'text+image' — tracks which pages needed escalation
        llm_extraction_raw STRING,
        extracted_at TIMESTAMP
    )
""").result()

print("Silver carrierOfferGrid table ready.")

GRID_TEXT_FIRST_PAGE_TYPES = [
    "postpaid_bts_offers",
    "cable_mvno_offers",
    "business_device_offers",
    "prepaid_brand_offers",
]

CARRIER_GRID_SCHEMA = {
    "type": "object",
    "properties": {
        "offers": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "carrier": {"type": "string"},
                    "item_name": {"type": "string"},
                    "item_group": {"type": "string"},
                    "plan_tier": {"type": "string"},
                    "offer_type": {"type": "string"},
                    "offer_amount": {"type": "string"},
                    "offer_condition": {"type": "string"},
                },
                "required": ["carrier", "item_name", "item_group"],
            },
        }
    },
    "required": ["offers"],
}

CARRIER_GRID_TEXT_PROMPT = """You are extracting structured offer data from a carrier \
comparison table. There is no image for this page — work from the text below, which \
is a column-aware reconstruction (each carrier's content is grouped together, marked \
by [COLUMN N] where applicable).

IMPORTANT — the same two patterns seen on other pages in this report apply here too:
1. One offer can apply to several items at once (e.g. one bullet covering several \
watch models or device tiers). Emit ONE RECORD PER ITEM, repeating the shared values, \
and set item_group to the full comma-separated list of items that share it (including \
the item itself). For a standalone offer, item_group should just equal item_name.
2. One item can have different prices per rate plan tier (e.g. Good/Better/Best). Emit \
ONE RECORD PER TIER, with plan_tier set to that tier's name and offer_amount set to \
only that tier's price. Do not combine multiple tier prices into one offer_amount string.

Known carriers from prior reports — use these exact names when the offer is from one \
of them, for naming consistency across weeks: {known_carriers}

For each carrier + item (+ tier, if applicable) combination with a stated offer, \
extract one record: carrier, item_name (exactly one item, e.g. device/watch/plan \
name), item_group, plan_tier (if applicable, else blank), offer_type (your best \
label), offer_amount (for exactly this one record), offer_condition (the requirement \
text). Skip entries with no stated offer ("No offer", "--", "N/A").

TEXT:
{page_text}"""


def extract_carrier_grid_text(page_number, page_type, page_text, report_date, run_id, known_carriers):
    prompt = CARRIER_GRID_TEXT_PROMPT.format(
        page_text=page_text,
        known_carriers=", ".join(known_carriers) if known_carriers else "(none yet — first run)",
    )
    result = call_gemini_json(prompt, CARRIER_GRID_SCHEMA)
    offers = result.get("offers", [])

    seen, deduped_offers = set(), []
    for o in offers:
        sig = (o.get("carrier"), o.get("item_name"), o.get("plan_tier"), o.get("offer_amount"))
        if sig not in seen:
            seen.add(sig)
            deduped_offers.append(o)
    if len(deduped_offers) < len(offers):
        print(f"  (removed {len(offers) - len(deduped_offers)} exact-duplicate record(s) from model response)")

    rows, review_rows = [], []
    for o in deduped_offers:
        carrier = o.get("carrier")
        amount_ok = values_present_in_text(o, ["offer_amount"], page_text)
        rows.append({
            "report_date": report_date.isoformat(),
            "run_id": run_id,
            "source_page": page_number,
            "page_type": page_type,
            "carrier": carrier,
            "item_name": o.get("item_name"),
            "item_group": o.get("item_group"),
            "plan_tier": o.get("plan_tier"),
            "offer_type": o.get("offer_type"),
            "offer_amount": o.get("offer_amount"),
            "offer_condition": o.get("offer_condition"),
            "values_agree": amount_ok,
            "extraction_method": "text",
            "llm_extraction_raw": json.dumps(result),
            "extracted_at": datetime.utcnow().isoformat(),
        })

        if carrier and known_carriers and carrier not in known_carriers:
            review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
                "source_table": "silver_carrierOfferGrid", "flag_reason": "new_carrier",
                "carrier": carrier, "item_name": o.get("item_name"), "offer_amount": o.get("offer_amount"),
                "detail": f"'{carrier}' not in known carrier list.", "status": "pending",
                "flagged_at": datetime.utcnow().isoformat(),
            })
        elif not amount_ok:
            review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
                "source_table": "silver_carrierOfferGrid", "flag_reason": "amount_mismatch",
                "carrier": carrier, "item_name": o.get("item_name"), "offer_amount": o.get("offer_amount"),
                "detail": "offer_amount doesn't appear verbatim in the page's extracted text.",
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })

    return rows, review_rows


def run_carrier_grid_extraction(run_id, report_date):
    print(f"[{run_id}] Carrier grid (text-first) extraction starting...")
    known_carriers = get_known_carriers()
    all_rows, all_review_rows = [], []

    pages = fetch_bronze_pages(run_id, GRID_TEXT_FIRST_PAGE_TYPES)
    if not pages:
        print("  No matching bronze pages found for this run_id — check show_bronze_state().")
        return

    for page in pages:
        print(f"\n--- Page {page['page_number']} ({page['page_type']}) ---")
        try:
            rows, review_rows = extract_carrier_grid_text(
                page["page_number"], page["page_type"], page["text_extraction_raw"],
                report_date, run_id, known_carriers,
            )
            print(f"  extracted {len(rows)} records ({len(review_rows)} flagged for review)")
            print_extraction_results(rows, page["text_extraction_raw"])
            all_rows.extend(rows)
            all_review_rows.extend(review_rows)
        except Exception as e:
            print(f"  FAILED: {e}")
            all_review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page["page_number"],
                "source_table": "silver_carrierOfferGrid", "flag_reason": "extraction_failed",
                "carrier": None, "item_name": None, "offer_amount": None, "detail": str(e),
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })

    if all_rows:
        delete_silver_rows_for_date(SILVER_GRID_TABLE, report_date)
        write_rows(SILVER_GRID_TABLE, all_rows)
        print(f"\n[{run_id}] Wrote {len(all_rows)} rows to silver carrierOfferGrid (prior rows for this date cleared first).")
    if all_review_rows:
        write_review_rows(all_review_rows)
        print(f"[{run_id}] Flagged {len(all_review_rows)} row(s) for review.")
        print_flag_breakdown(all_review_rows)

# run_carrier_grid_extraction(run_id="...", report_date=date(2026, 6, 19))